# Estudo de Extração de Features

Demonstra o **front-end real** usado pelo sistema na inferência
(`app.domain.services.detection.audio_preprocessing.prepare_audio_for_model`,
baseado em `tf.signal`, in-graph):

- **LFCC** (default desde a melhoria P0 — supera o mel em anti-spoofing);
- **log-mel**;
- **raw-audio** (forma de onda PCM 1D);
- métricas clássicas de estudo: **MFCC**, **centroide espectral**,
  **bandwidth**, **ZCR** e energia RMS.

Configuração: `sample_rate=16 kHz`, `n_fft=512`, `hop_length=128`,
`n_mels = n_lfcc = 80`.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

_REPO_URL = "https://github.com/thierrybraga/XFakeSong.git"


def _project_root() -> Path:
    p = Path.cwd()
    while not (p / "app").exists() and p.parent != p:
        p = p.parent
    return p


# Colab/Kaggle limpos: clona o repositório se o projeto não estiver no disco.
if not (_project_root() / "app").exists():
    if not Path("XFakeSong").exists():
        print("Clonando XFakeSong...")
        subprocess.run(["git", "clone", "--depth", "1", _REPO_URL], check=True)
    os.chdir("XFakeSong")

_DEPS = {"librosa": "librosa>=0.10", "soundfile": "soundfile>=0.12",
         "pywt": "PyWavelets>=1.4"}
_missing = [s for m, s in _DEPS.items() if importlib.util.find_spec(m) is None]
if _missing:
    print("Instalando dependências de áudio:", *_missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing],
                   check=True)

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT, "| Colab:", "google.colab" in sys.modules)

## 1. Gerar um sinal sintético de 1 segundo

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
t = np.linspace(0, 1.0, 16000, endpoint=False)
# Soma de senóides + ruído leve (proxy de voz para demonstração).
wav = (0.6 * np.sin(2 * np.pi * 220 * t)
       + 0.3 * np.sin(2 * np.pi * 440 * t)
       + 0.05 * rng.standard_normal(16000)).astype("float32")
print("waveform:", wav.shape, wav.dtype)

## 2. Extrair os três front-ends reais

In [ ]:
from app.domain.services.detection.audio_preprocessing import (
    prepare_audio_for_model,
)

lfcc = prepare_audio_for_model(
    wav, input_type="spectrogram", input_shape=(100, 80),
    sample_rate=16000, feature_frontend="lfcc",
)
logmel = prepare_audio_for_model(
    wav, input_type="spectrogram", input_shape=(100, 80),
    sample_rate=16000, feature_frontend="logmel",
)
raw = prepare_audio_for_model(
    wav, input_type="raw_audio", input_shape=(16000, 1),
    sample_rate=16000,
)
print("LFCC   :", np.asarray(lfcc).shape)
print("log-mel:", np.asarray(logmel).shape)
print("raw    :", np.asarray(raw).shape)

## 3. Features clássicas para estudo

In [ ]:
import librosa
import pandas as pd

mfcc = librosa.feature.mfcc(y=wav, sr=16000, n_mfcc=13)
centroid = librosa.feature.spectral_centroid(y=wav, sr=16000)
bandwidth = librosa.feature.spectral_bandwidth(y=wav, sr=16000)
zcr = librosa.feature.zero_crossing_rate(wav)
rms = librosa.feature.rms(y=wav)

feature_summary = pd.DataFrame([{
    "mfcc_mean": float(mfcc.mean()),
    "mfcc_std": float(mfcc.std()),
    "centroid_mean_hz": float(centroid.mean()),
    "bandwidth_mean_hz": float(bandwidth.mean()),
    "zcr_mean": float(zcr.mean()),
    "rms_mean": float(rms.mean()),
}])
feature_summary

## 4. Visualizar

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 3, figsize=(13, 6.2))
ax = ax.ravel()
ax[0].plot(wav[:1000], color="#3b82f6", lw=0.8)
ax[0].set_title("Forma de onda (1000 amostras)")
ax[1].imshow(np.asarray(lfcc).T, aspect="auto", origin="lower", cmap="magma")
ax[1].set_title("LFCC (80 × T)")
ax[2].imshow(np.asarray(logmel).T, aspect="auto", origin="lower", cmap="magma")
ax[2].set_title("log-mel (80 × T)")
ax[3].imshow(mfcc, aspect="auto", origin="lower", cmap="viridis")
ax[3].set_title("MFCC")
ax[4].plot(centroid.ravel(), label="centroid")
ax[4].plot(bandwidth.ravel(), label="bandwidth")
ax[4].set_title("Centroide e bandwidth")
ax[4].legend()
ax[5].plot(zcr.ravel(), label="ZCR")
ax[5].plot(rms.ravel(), label="RMS")
ax[5].set_title("ZCR e RMS")
ax[5].legend()
fig.tight_layout()
plt.show()

## Leituras

- `docs/04_FEATURES.md` — features e extratores.
- `docs/09_INFERENCIA.md` — como o `input_contract` garante paridade
  treino↔inferência (o mesmo front-end usado aqui é reproduzido na
  detecção).